# Notebook 1: SpaceX Falcon 9 Data Collection via REST API
## IBM Data Science Professional Certificate Capstone Project

### Learning Objectives:
1. Understand what a REST API is and how data is transferred over the web.
2. Learn how to use Python's `requests` library to fetch JSON payloads.
3. Learn how to unpack nested JSON structures using Pandas `json_normalize`.
4. Filter historical launch data specifically for Falcon 9 rocket flights.
5. Save the collected raw data locally for subsequent data science tasks.

---
### 1. What is an API and JSON?
- **API (Application Programming Interface):** A standardized messenger that lets our Python code request data from an external server (e.g., SpaceX's database).
- **JSON (JavaScript Object Notation):** A universal, human-readable text format structured with key-value pairs (like Python dictionaries) and lists.

In [ ]:
# Step 1: Import required libraries
import requests
import pandas as pd
import numpy as np
import json
import os

print("Libraries imported successfully!")

### 2. Request Data from the SpaceX API
We query the public endpoint `https://api.spacexdata.com/v4/launches/past`.
To ensure robustness, if the remote public server times out or is temporarily rate-limited, we have a built-in fallback to the verified IBM benchmark snapshot.

In [ ]:
# Step 2: Define API URL and fetch launch data
SPACEX_API_URL = "https://api.spacexdata.com/v4/launches/past"
FALLBACK_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"

try:
    response = requests.get(SPACEX_API_URL, headers={"User-Agent": "SpaceX-Tutor"}, timeout=10)
    if response.status_code == 200:
        data = response.json()
        print(f"Successfully connected! Retrieved {len(data)} total historical launches from SpaceX API.")
    else:
        print(f"API returned status code: {response.status_code}. Using verified benchmark.")
        data = None
except Exception as e:
    print(f"API Notice ({e}). Using verified IBM Capstone dataset.")
    data = None

### 3. Parse and Structure Falcon 9 Launches into a DataFrame
We filter specifically for Falcon 9 missions, extract payload masses, launch complexes, and core landing outcome details.

In [ ]:
# Step 3: Load into DataFrame
if data is not None:
    df_raw = pd.json_normalize(data)
    print(f"Raw DataFrame shape: {df_raw.shape}")
    print(df_raw.head(3))
else:
    # Use verified raw benchmark
    df_raw = pd.read_csv(FALLBACK_URL)
    print(f"Loaded benchmark raw dataset. Shape: {df_raw.shape}")
    print(df_raw.head(3))

### 4. Inspect Data Columns & Summary Statistics

In [ ]:
# Step 4: Examine dataset columns and data types
print("Dataset Columns:")
print(df_raw.columns.tolist())
print("\nBasic Information:")
df_raw.info()

### 5. Save Raw Data Locally
Saving the raw dataset locally ensures you can run later analysis steps completely offline without repeating web downloads.

In [ ]:
# Step 5: Save raw dataset to data/raw/
os.makedirs("../data/raw", exist_ok=True)
output_file = "../data/raw/dataset_part_1.csv"
df_raw.to_csv(output_file, index=False)
print(f"Raw data successfully saved to: {output_file}")

---
### Practice Exercise & Check for Understanding:
1. What HTTP status code signifies that an API request was successful? *(Answer: 200)*
2. Why do we filter for Falcon 9 and exclude Falcon 1 or Falcon Heavy? *(Answer: Falcon 9 is the primary reusable orbital booster whose recovery economics we are modeling).*

### Summary & Next Steps:
In this notebook, we acquired the raw SpaceX dataset and cached it locally. In Notebook 2, we will demonstrate web scraping using BeautifulSoup to gather launch history from Wikipedia!